# Subsetting original VCF for demographic modeling

### Imports

In [4]:
import dadi
import os
import pandas as pd
import vcf
import numpy as np
import shutil
import pysam
import random
import subprocess
import gzip

In [5]:
%pwd

'/n/holylfs05/LABS/hopkins_lab/Users/pfmckenzie/projects/ddrad_demography'

In [6]:
def extract_locations_from_vcf(vcf_file):
    """
    returns all positions in a vcf
    """
    
    # open VCF
    vcf = pysam.VariantFile(vcf_file)

    # init list to store locations
    locations = []

    # iter over the vcf
    for record in vcf:
        # save position
        contig = record.chrom
        position = record.pos

        # save tuple to locations list
        locations.append((contig, position))
    return np.array(locations)

# Sample SNPs -- one per locus. We can use a gap size threshold to enforce a required distance between loci on the same chromosome.

### Helper functions

In [7]:
def sample_locations(array, gap_size=50):
    """
    return list of RAD loci separated by gaps larger than the min 'gap_size'
    if contig has only one locus, samples that locus

    :param array: np array of contig names and positions
    :param gap_size: gap size threshold defining separate loci
    :return: list of sampled locations
    """
    #sampled_locations = []
    previous_contig = None
    previous_position = None
    current_locus = []
    loci = []

    for contig, position in array:
        position = int(position)

        # check if we moved to a new contig or if the gap is larger than the threshold
        if contig != previous_contig or (previous_position is not None and position - previous_position > gap_size):
            if current_locus:
                start = min([i[1] for i in current_locus])
                stop = max([i[1] for i in current_locus])
                loci.append([previous_contig, start, stop])
                
                # reset
                current_locus = []

        # Update the current locus and previous position
        current_locus.append((contig, position))
        previous_contig, previous_position = contig, position

    # samp from the last locus if it's not empty
    if current_locus:
        start = min([i[1] for i in current_locus])
        stop = max([i[1] for i in current_locus])
        loci.append([previous_contig, start, stop])

    return loci

def sample_variant_locus(vcf, ## open pysam object
                         target_contig,
                         start_position,
                         stop_position):
    '''
    this is the function that, given a pysam vcf object,
    a target contig, and start/stop positions for a locus,
    will return the positions of snps from that locus
    '''
    # init list to store positions
    positions = []
    
    # iter over the target region for positions in vcf
    for record in vcf.fetch(target_contig, start_position, stop_position):
        # add any vcf positions to list
        positions.append(record.pos)
    return(positions)

def sample_snps(snps_vcf_path,
                sampled_locations
               ):
    '''
    this wraps around the above function.
    It accepts a path to a snps vcf file, as well
    as a list of the locations of rad loci to sample
    snps from (one snp per locus).
    It returns a list of sampled snp locations

    :param sampled_locations: list returned by `sample_locations`,
        for which each element has three values: a contig id, start, 
        and stop, which denote location of a rad locus
    '''
    sampled_snps = []
    
    # open vcf
    vcf_ = pysam.VariantFile(snps_vcf_path)

    # iterate over all of the positions
    for loc in sampled_locations:
        snps_in_locus = sample_variant_locus(
            vcf_,
            loc[0],
            loc[1],
            loc[2]
        )
        if snps_in_locus:
            sampled_snps.append([loc[0], np.random.choice(snps_in_locus)])
    
    # close vcf
    vcf_.close()
    return(sampled_snps)

### Run the helper functions to sample one snp per locus

In [19]:
# gvcf
vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/populations.all.vcf"
# snps vcf
snps_vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/populations_sorted.snps.vcf.gz"

# get all sites with data in gvcf
locations = extract_locations_from_vcf(vcf_path)

# get locations of all rad loci
sampled_locations = sample_locations(locations)

# sample one snp per rad locus
selected_snps = sample_snps(snps_vcf_path,sampled_locations)
len(selected_snps)

53325

### Write the sampled snps out to a file

In [17]:
one_snp_loc_file_path = '/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_per_loc.vcf.gz'

# input: vcf of all snp data
vcf_in = pysam.VariantFile(snps_vcf_path)

# output: vcf with only one snp per rad locus
vcf_out = pysam.VariantFile(one_snp_loc_file_path, 'w', header=vcf_in.header)

# iter over list of sampled snps
for contig, pos in selected_snps:
    try:
        # get that specific position from input vcf
        for record in vcf_in.fetch(contig, pos - 1, pos):
            if record.pos == pos:
                # write that position out to the output vcf
                vcf_out.write(record)
    except ValueError:
        # helpful for debugging
        print(f"No record found for {contig}:{pos}")

# close vcfs
vcf_in.close()
vcf_out.close()

print(f"Filtered VCF file created: {one_snp_loc_file_path}")

[W::hts_idx_load3] The index file is older than the data file: /n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/populations.snps.lFilt.iFilt.vcf.gz.csi


Filtered VCF file created: /n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_per_loc.vcf.gz


# Cleaning steps:

### Filter singletons

In [18]:
def filter_vcf(input_vcf, output_vcf):
    try:
        # bcftools command to filter singletons
        cmd = [
            'bcftools', 'view', 
            '-i', 'AC!=1', # no variants where AC is 1
            '-o', output_vcf,
            '-O', 'z', # z is compressed VCF
            input_vcf,
        ]
        
        # run
        subprocess.run(cmd, check=True)
        print(f"Filtered VCF saved to {output_vcf}")
        
    except subprocess.CalledProcessError as e:
        print(f"An error occurred while filtering the VCF: {e}")

# run it:
input_vcf = one_snp_loc_file_path
output_vcf = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_populations.snps.vcf.gz"
filter_vcf(input_vcf, output_vcf)


Filtered VCF saved to /n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_populations.snps.vcf.gz


### Flip strands

In [19]:
def process_vcf(ddrad_vcf_path, out_vcf_path, ref_path, awk_script_path, log_path="process.log"):
    with open(log_path, 'w') as log_file, open(out_vcf_path, 'wb') as out_file:
        cmd1 = ["bcftools", "view", ddrad_vcf_path] # init chain with vcf
        cmd2 = ["awk", "-f", awk_script_path] # flip strand
        cmd3 = ["bcftools", "norm", "-c", "s", "-f", ref_path] # re-check against ref
        cmd4 = ["bgzip", "-c"]

        p1 = subprocess.Popen(cmd1, stdout=subprocess.PIPE, stderr=log_file)
        p2 = subprocess.Popen(cmd2, stdin=p1.stdout, stdout=subprocess.PIPE, stderr=log_file)
        p3 = subprocess.Popen(cmd3, stdin=p2.stdout, stdout=subprocess.PIPE, stderr=log_file)
        p4 = subprocess.Popen(cmd4, stdin=p3.stdout, stdout=out_file, stderr=log_file)  # output directed to out_file
        
        _, errs = p4.communicate()

        if p4.returncode == 0:
            print(f"Successfully processed the VCF file. Output saved at: {out_vcf_path}")
        else:
            print(f"An error occurred during processing. Check {log_path} for details.")

# no singletons vcf
input_vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_populations.snps.vcf.gz"
output_vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_flipped_populations.snps.vcf.gz"
awk_path = "/n/holyscratch01/hopkins_lab/Everyone/felixw/homemade/scripts/flip_strand.ddrad.awk"
ref_path = "./assemblies/phlox_pilo.v1.fasta"

# run it:
process_vcf(input_vcf_path, output_vcf_path, ref_path, awk_path)

Successfully processed the VCF file. Output saved at: /n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_flipped_populations.snps.vcf.gz


### Add an "ancestral allele" annotation to INFO

In [20]:
def get_fasta_sequence(chrom, pos, fasta_path):
    cmd = ["samtools", "faidx", fasta_path, f"{chrom}:{pos}-{pos}"]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, text=True)
    sequence = result.stdout.split('\n')[1]
    return sequence.upper()

def annotate_vcf_with_ancestral_allele(vcf_path, fasta_path, output_vcf_path):
    open_func = gzip.open if vcf_path.endswith(".gz") else open
    with open_func(vcf_path, 'rt') as vcf, gzip.open(output_vcf_path, 'wt') as output_vcf:
        for line in vcf:
            if line.startswith("#"):
                output_vcf.write(line)
            else:
                columns = line.strip().split("\t")
                chrom = columns[0]
                pos = columns[1]
                info = columns[7]
                
                ancestral_allele = get_fasta_sequence(chrom, pos, fasta_path)
                
                if "AA=" not in info:
                    new_info = f"{info};AA={ancestral_allele}"
                    columns[7] = new_info
                
                output_vcf.write("\t".join(columns) + "\n")

In [21]:
input_vcf = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_flipped_populations.snps.vcf.gz"
output_vcf = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_polarized_flipped_populations.snps.vcf.gz"

annotate_vcf_with_ancestral_allele(input_vcf, ref_path, output_vcf)

### Follow-up: Add the info tag to the header for AA

In [22]:
def add_AA_definition_to_vcf_header(input_vcf_path, output_vcf_path):
    open_func = gzip.open if input_vcf_path.endswith(".gz") else open
    write_func = gzip.open if output_vcf_path.endswith(".gz") else open
    
    with open_func(input_vcf_path, 'rt') as vcf, write_func(output_vcf_path, 'wt') as output_vcf:
        for line in vcf:
            if line.startswith("#CHROM"):
                # put in the def of the AA INFO tag before the #CHROM line
                output_vcf.write('##INFO=<ID=AA,Number=1,Type=String,Description="Ancestral Allele">\n')
            output_vcf.write(line)

# no singletons, strand-flipped, ancestral-annotated vcf
vcf_path = "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_polarized_flipped_populations.snps.vcf.gz"
add_AA_definition_to_vcf_header(vcf_path, 
                                "/n/home09/pfmckenzie/lab_scratch/Lab/ddrad_genome_vcfs/pilo_aligned/multi/cusp_drum_roem/res_stacks/min_samples_per_pop~0.1/min_samples_overall~0.1/mind_0.9/one_snp_no_singles_polarized_flipped_populations_AAinfo.snps.vcf.gz")